# Wafer Guard: Audit and Repair
Chronological evaluation of a wafer anomaly detector.
The test set is kept at its natural prevalence.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, average_precision_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'raw' / 'secom_telemetry_modified.csv'
torch.manual_seed(42)
np.random.seed(42)
print('Data path:', DATA_PATH)

Data path: A:\05_Wafer\wafer-guard\data\raw\secom_telemetry_modified.csv


In [2]:
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='raise')
df = df.sort_values('timestamp', kind='mergesort').reset_index(drop=True)
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
n = len(df)
a, b = int(n * 0.70), int(n * 0.80)
train_df, val_df, test_df = df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()
print('Rows:', len(train_df), len(val_df), len(test_df))
print('Test defect prevalence:', (test_df['label'] == 1).mean())

Rows: 1096 157 314
Test defect prevalence: 0.054140127388535034


## PS-L handling
For rows where `sensor_0 == 0.0`, the value is a sentinel. For those rows, `sensor_1` is categorical rather than continuous.

In [3]:
def get_features(part):
    X = part[sensor_cols].copy()
    legacy = X['sensor_0'].eq(0.0).fillna(False)
    X['is_legacy_ps_l'] = legacy.astype(float)
    X.loc[legacy, 'sensor_0'] = np.nan
    sensor1 = X['sensor_1'].copy()
    X['sensor_1'] = sensor1.mask(legacy, np.nan)
    for code in (100.0, 200.0, 300.0):
        X[f'legacy_sensor_1_code_{int(code)}'] = (legacy & sensor1.eq(code)).astype(float)
    y = (part['label'] == 1).astype(np.int64).to_numpy()
    return X, y
X_train_raw, y_train = get_features(train_df)
X_val_raw, y_val = get_features(val_df)
X_test_raw, y_test = get_features(test_df)

In [4]:
numeric_cols = [c for c in sensor_cols if c != 'sensor_1']
usable_cols = [c for c in numeric_cols if X_train_raw[c].notna().any() and X_train_raw[c].nunique(dropna=True) > 1]
usable_cols.append('sensor_1')
flag_cols = ['is_legacy_ps_l', 'legacy_sensor_1_code_100', 'legacy_sensor_1_code_200', 'legacy_sensor_1_code_300']
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
train_imputed = imputer.fit_transform(X_train_raw[usable_cols])
val_imputed = imputer.transform(X_val_raw[usable_cols])
test_imputed = imputer.transform(X_test_raw[usable_cols])
train_scaled = scaler.fit_transform(train_imputed)
val_scaled = scaler.transform(val_imputed)
test_scaled = scaler.transform(test_imputed)
def combine_features(scaled, raw):
    return np.column_stack([scaled, raw[flag_cols].to_numpy(dtype=np.float32)]).astype(np.float32)
X_train = combine_features(train_scaled, X_train_raw)
X_val = combine_features(val_scaled, X_val_raw)
X_test = combine_features(test_scaled, X_test_raw)
print('Feature shapes:', X_train.shape, X_val.shape, X_test.shape)
print('All finite:', np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all())

Feature shapes: (1096, 472) (157, 472) (314, 472)
All finite: True


## Model and shape verification
Input and reconstruction tensors must both have shape `[batch_size, features]`.

In [5]:
class WaferAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, latent_dim), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, input_dim))
        self.classifier = nn.Linear(latent_dim, 1)
    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(f'Expected a 2D batch, got {tuple(x.shape)}')
        z = self.encoder(x)
        return self.decoder(z), self.classifier(z).squeeze(-1), z
model = WaferAutoencoder(X_train.shape[1])
sample = torch.tensor(X_train[:8])
reconstructed, logits, latent = model(sample)
print('Input:', tuple(sample.shape))
print('Reconstruction:', tuple(reconstructed.shape))
print('Logits:', tuple(logits.shape))
assert reconstructed.shape == sample.shape
assert logits.shape == (len(sample),)

Input: (8, 472)
Reconstruction: (8, 472)
Logits: (8,)


In [6]:
train_data = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32))
loader = DataLoader(train_data, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
positive_count = max(int(y_train.sum()), 1)
positive_weight = torch.tensor([(len(y_train) - positive_count) / positive_count])
bce = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
mse = nn.MSELoss()
for epoch in range(20):
    model.train()
    total = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        reconstructed, logits, _ = model(batch_x)
        loss = mse(reconstructed, batch_x) + bce(logits, batch_y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(batch_x)
    print(f'Epoch {epoch + 1:02d}/20: {total / len(train_data):.4f}')

Epoch 01/20: 2.2726
Epoch 02/20: 2.1451
Epoch 03/20: 1.8276


Epoch 04/20: 1.4421
Epoch 05/20: 1.1632
Epoch 06/20: 1.0318


Epoch 07/20: 0.9743
Epoch 08/20: 0.9434
Epoch 09/20: 0.9380


Epoch 10/20: 0.9945
Epoch 11/20: 0.9092


Epoch 12/20: 0.8691
Epoch 13/20: 0.8768
Epoch 14/20: 0.8721


Epoch 15/20: 0.8403
Epoch 16/20: 0.8155


Epoch 17/20: 0.8081
Epoch 18/20: 0.8020
Epoch 19/20: 0.7886


Epoch 20/20: 0.7796


In [7]:
model.eval()
with torch.no_grad():
    val_prob = torch.sigmoid(model(torch.tensor(X_val))[1]).numpy()
thresholds = np.arange(0.05, 0.96, 0.05)
threshold = max(thresholds, key=lambda t: f1_score(y_val, (val_prob >= t).astype(int), zero_division=0))
with torch.no_grad():
    test_prob = torch.sigmoid(model(torch.tensor(X_test))[1]).numpy()
test_pred = (test_prob >= threshold).astype(int)
print('Threshold:', float(threshold))
print('Test prevalence:', float(y_test.mean()))
print('Accuracy:', accuracy_score(y_test, test_pred))
print('Precision:', precision_score(y_test, test_pred, zero_division=0))
print('Recall:', recall_score(y_test, test_pred, zero_division=0))
print('F1:', f1_score(y_test, test_pred, zero_division=0))
print('Average precision:', average_precision_score(y_test, test_prob))
print('ROC-AUC:', roc_auc_score(y_test, test_prob) if len(np.unique(y_test)) == 2 else 'unavailable')
print('Confusion matrix:\n', confusion_matrix(y_test, test_pred))

Threshold: 0.05
Test prevalence: 0.054140127388535034
Accuracy: 0.9426751592356688
Precision: 0.0
Recall: 0.0
F1: 0.0
Average precision: 0.09477600504850578


ROC-AUC: 0.6324024559318677
Confusion matrix:
 [[296   1]
 [ 17   0]]


## Audit conclusion
The chronological test set is evaluated without balancing. Report the measured metrics from the executed cells. A high accuracy alone is not sufficient for a rare-defect detector; recall and precision must also be considered. Do not deploy unless defect detection performance is acceptable.